# 02 — Curve Construction Validation
Visual inspection of interpolated term structures and curve metrics.
Run after `make curves`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from commodity_curve_factors.curves.builder import load_curves
from commodity_curve_factors.curves.metrics import (
    compute_carry, compute_slope, compute_curvature, compute_term_carry
)
from commodity_curve_factors.utils.config import load_config

curves = load_curves()
curve_config = load_config("curve")
tenors = [f"F{m}M" for m in curve_config["standard_tenors"]]

print(f"Loaded curves for {len(curves)} commodities")
for sym, c in sorted(curves.items()):
    print(f"  {sym}: {len(c)} days, {c.isna().mean().max():.1%} max NaN")

In [ ]:
notable_dates = {
    "2008-07-03": "WTI peak ($146)",
    "2015-01-15": "Shale glut contango",
    "2020-04-20": "COVID negative front",
    "2022-03-01": "Russia war backwardation",
}

cl = curves["CL"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (date_str, label) in zip(axes.flatten(), notable_dates.items()):
    ts = pd.Timestamp(date_str)
    if ts in cl.index:
        row = cl.loc[ts, tenors]
        tenor_months = curve_config["standard_tenors"]
        ax.plot(tenor_months, row.values, "o-", markersize=5)
        ax.set_xlabel("Months to Expiry")
        ax.set_ylabel("Price ($)")
        ax.set_title(f"{date_str}: {label}")
        ax.grid(True, alpha=0.3)
    else:
        ax.set_title(f"{date_str}: NOT IN DATA")

plt.suptitle("WTI Term Structure Snapshots", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
date = pd.Timestamp("2022-03-01")
fig, axes = plt.subplots(3, 2, figsize=(12, 12))

for ax, sym in zip(axes.flatten(), ["CL", "NG", "GC", "ZC", "KC", "HG"]):
    if sym in curves and date in curves[sym].index:
        row = curves[sym].loc[date, tenors]
        ax.plot(curve_config["standard_tenors"], row.values, "o-", markersize=5)
        ax.set_title(f"{sym} — {date.date()}")
        ax.set_xlabel("Months")
        ax.set_ylabel("Price")
        ax.grid(True, alpha=0.3)
    else:
        ax.set_title(f"{sym} — NOT AVAILABLE")

plt.suptitle("Cross-Commodity Term Structures", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
cl = curves["CL"]
carry = compute_carry(cl)
slope = compute_slope(cl)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(carry.index, carry, linewidth=0.5)
ax1.axhline(0, color="gray", linestyle="--", linewidth=0.5)
ax1.set_title("CL Carry: (F1M - F2M) / F2M x 12")
ax1.set_ylabel("Annualized Carry")

ax2.plot(slope.index, slope, linewidth=0.5, color="tab:orange")
ax2.axhline(0, color="gray", linestyle="--", linewidth=0.5)
ax2.set_title("CL Slope: (F12M - F1M) / F1M")
ax2.set_ylabel("Slope")

plt.tight_layout()
plt.show()

In [ ]:
from commodity_curve_factors.curves.metrics import compute_all_metrics

metrics = compute_all_metrics(curves)
carry_wide = metrics["carry"]

# Resample to monthly for readability
monthly = carry_wide.resample("ME").mean()

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.pcolormesh(
    monthly.index, range(len(monthly.columns)), monthly.T.values,
    cmap="RdYlGn", vmin=-0.3, vmax=0.3, shading="auto",
)
ax.set_yticks(range(len(monthly.columns)))
ax.set_yticklabels(monthly.columns)
ax.set_title("Monthly Carry Heatmap (green = backwardation, red = contango)")
plt.colorbar(im, ax=ax, label="Carry")
plt.tight_layout()
plt.show()

In [ ]:
nan_summary = pd.DataFrame({
    sym: curves[sym][tenors].isna().mean() * 100
    for sym in sorted(curves.keys())
}).T
nan_summary.columns.name = "Tenor"
print("NaN % by commodity x tenor:")
print(nan_summary.round(1).to_string())